# Kaggle: текст → география и география → текст

Один корпус из train.parquet, одинаковые val/test-контексты и Recall@10/50/100.
1. Top-N каждого источника → union item_id → расстояние (RRF разрешает ничьи).
2. Ближайшие N → точные скоры всех моделей → лексикографический кортеж SCORE_ORDER → расстояние.

В кортежe следующий скор влияет только при равенстве предыдущего. Это не усреднение моделей.
Модели: BM25 с лемматизацией по заголовку, описанию, параметрам; Giga и BGE по заголовку.
FAISS содержит объявления; запросы кодируются на GPU при retrieve/score_candidates.
Каждая модель загружается отдельно. Кешируются выдачи и скоры пар, а не query-векторы.

Фичи проверяем отдельно и вместе через поздний rerank. Val выбирает N для каждой базы,
затем веса фич при том же N. Test не участвует в выборе. Корпус общий, центры только по train.
Предыдущие эксперименты уже использовали test: это исследовательское сравнение.

In [ ]:
from pathlib import Path
REPO_URL = ""  # URL вашего репозитория с актуальными geo_first.py/hybrid.py
REPO_REF = None  # необязательно: коммит/ветка для клонирования
REPO_PATH = None  # существующий репозиторий; None — поиск, затем clone
DATA_PATH = None  # полный путь к train.parquet при неоднозначном автопоиске
DENSE_INDEX_DIR = None  # None — auto-detect attached indices or build under /kaggle/working
INSTALL_DEPENDENCIES = True
DENSE_MODELS = ["giga", "bge"]  # [] для проверки только BM25
SEED = 42
SAMPLE_SEED = 43
N_VAL = 500
N_TEST = 5000
NS = [50, 500, 5000]
KS = [10, 50, 100]
SELECT_K = 50
QUERY_BATCH_SIZE = 8
DENSE_BATCH_SIZE = 16
DENSE_MAX_LENGTH = 128  # как в прежнем title-эксперименте
BUILD_DENSE_INDEXES = True  # кодировать объявления на Kaggle GPU, сохранить для повторного запуска
BM25_LEMMATIZE = True
SCORE_ORDER = ["bm25_title", "giga", "bge", "bm25_description", "bm25_infm"]
QUALITY_DEPTH = 300
QUALITY_WEIGHTS = [0.0, 0.01, 0.05, 0.1]
ARTIFACTS = Path("/kaggle/working/retrieval_comparison")

## Окружение и индексы

В Kaggle включите GPU и Internet. Прикрепите train.parquet и актуальный код репозитория. Clone идёт в отдельную папку; артефакты — отдельно.
Если title-индексов Giga/BGE ещё нет, ноутбук построит их на Kaggle GPU и сохранит в DENSE_INDEX_DIR.
При повторном запуске готовые индексы переиспользуются. Корпус кодируется по одной модели за раз.

In [ ]:
import sys, subprocess, json, hashlib, gc
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "bm25s>=0.3.11", "pymorphy3>=2", "razdel>=0.5", "polars>=1.44,<2",
        "pandas>=2", "scikit-learn>=1.6,<2", "faiss-cpu>=1.10,<2",
        "sentence-transformers>=5.1,<6", "transformers>=4.51,<5"])
roots = [Path.cwd(), *Path.cwd().parents, *Path("/kaggle/working").glob("*")]
ROOT = Path(REPO_PATH) if REPO_PATH else next((p for p in roots if (p / "src/retrieval_comparison.py").is_file()), None)
if ROOT is None:
    if not REPO_URL:
        raise ValueError("Укажите REPO_URL для clone или REPO_PATH на актуальный код.")
    ROOT = Path("/kaggle/working/comparison_repo")
    subprocess.check_call(["git", "clone", REPO_URL, str(ROOT)])
    if REPO_REF:
        subprocess.check_call(["git", "-C", str(ROOT), "checkout", REPO_REF])
ROOT = ROOT.resolve()
required_sources = [
    "src/retrieval_comparison.py", "src/geo.py",
    "src/retrievers/hybrid.py", "src/retrievers/geo_first.py",
]
missing_sources = [name for name in required_sources if not (ROOT / name).is_file()]
if missing_sources:
    raise FileNotFoundError(
        f"В {ROOT} нет актуальных файлов: {missing_sources}. "
        "Опубликуйте локальные изменения в Git и укажите эту ветку в REPO_REF, "
        "либо загрузите папку проекта в Kaggle Dataset и задайте REPO_PATH."
    )
sys.path.insert(0, str(ROOT))
from src.retrieval_comparison import RetrievalComparison
from IPython.display import display


## Корпус, сплит и дополнительные признаки

Пустые выдачи и запросы без центра остаются в знаменателе метрик. Во втором подходе
без центра нет геопула. Помимо Recall показываем потолок Recall пула и долю непустых выдач.

Рейтинг сглаживаем к среднему train с силой 20 отзывов; число отзывов — log1p с ограничением
по p99 train. Отсутствующие значения получают нейтральную поправку 0.
Телефон/сообщения: доступен канал → +1, запрещён → −1. Все четыре сигнала ограничены [-1,1].
Средние и масштабы вычисляются только по объявлениям train-части.

In [ ]:
experiment = RetrievalComparison(
    repo_path=ROOT,
    data_path=DATA_PATH,
    dense_index_dir=DENSE_INDEX_DIR,
    dense_models=DENSE_MODELS,
    seed=SEED,
    sample_seed=SAMPLE_SEED,
    n_val=N_VAL,
    n_test=N_TEST,
    ns=NS,
    ks=KS,
    select_k=SELECT_K,
    query_batch_size=QUERY_BATCH_SIZE,
    dense_batch_size=DENSE_BATCH_SIZE,
    dense_max_length=DENSE_MAX_LENGTH,
    build_dense_indices=BUILD_DENSE_INDEXES,
    bm25_lemmatize=BM25_LEMMATIZE,
    score_order=SCORE_ORDER,
    quality_depth=QUALITY_DEPTH,
    quality_weights=QUALITY_WEIGHTS,
    artifacts=ARTIFACTS,
)
experiment.prepare_data()
experiment.prepare_dense_indices()

## Кеш кандидатов и скоров

Одна модель за раз обслуживает оба подхода. Геопул строится один раз, для максимального N;
меньшие N берутся как префиксы. Аналогично кешируем максимальную текстовую выдачу.
Ключ кеша включает данные, запросы, настройки, индексы и код. Запись батчей атомарная.
Повторный запуск пропускает готовые батчи. Изменение N/моделей создаёт другой кеш.

In [ ]:
experiment.prepare_cache()
experiment.build()

## Два порядка ранжирования и фичи

Без фич используются точное расстояние / точный кортеж.
С фичами только top-300 базовой выдачи переставляется по
`1 − (rank−1)/(depth−1) + weight·feature`. Остальные позиции сохраняются.
Это контролируемая поправка к рангу, не сумма несопоставимых сырых скоров.
`all` — среднее четырёх сигналов. В сетке есть нулевой вес: можно отказаться от фичи.
Recall@50 не чувствителен к перестановкам строго внутри top-50, поэтому depth больше K.
N первого подхода — на источник: union может быть больше N второго подхода.
Потолок Recall и размер пула помогают учитывать разницу бюджетов.

## Val: N базы, затем вес каждого признака при том же N

Порядок моделей фиксирован в SCORE_ORDER до запуска. Для каждого подхода выбираем N по
Recall@50 без фич. Далее при этом N выбираем вес каждого признака; при ничьей — меньший вес.
Таблицы сохраняются вместе с конфигурацией эксперимента.

In [ ]:
validation, selected = experiment.validate()

## Test: фиксированные варианты и парные интервалы

Сравниваем каждую фичу с базой того же подхода и N. Bootstrap группируется по тексту
запроса: несколько контекстов одного текста не считаем независимыми наблюдениями.
Интервал, включающий 0, не подтверждает прирост. Значения per_query сохраняются для анализа.

In [ ]:
comparison = experiment.test()

## Финальный инференс на benchmark → answer.csv

Заполните плейсхолдеры лучшей конфигурацией **с val** и путями к benchmark-файлам.
RUN_BENCHMARK выключен по умолчанию. Обрабатываются все запросы benchmark_queries.parquet;
поиск идёт только по benchmark_items.parquet. Центры локаций и масштабы фич берём из train.

Dense-индексы для train не подходят: прикрепите отдельные benchmark-индексы либо включите
BUILD_BENCHMARK_INDEXES для их построения на Kaggle теми же моделями и настройками.
Индексы и батчи кешируются; держим в памяти одну модель за раз.

answer.csv: UTF-8, запятая, ровно query_id/answer, без индекса. ID сохраняются без изменения
регистра. Все запросы присутствуют один раз; до 50 уникальных item_id из benchmark на строку.
Пустая выдача сохраняется как пустая строка answer, запрос не пропускается.

In [ ]:
RUN_BENCHMARK = False
BENCHMARK_ITEMS_PATH = None    # TODO: /kaggle/input/.../benchmark_items.parquet
BENCHMARK_QUERIES_PATH = None  # TODO: /kaggle/input/.../benchmark_queries.parquet
BENCHMARK_INDEX_DIR = Path("/kaggle/working/benchmark_dense_indexes")
BUILD_BENCHMARK_INDEXES = False  # TODO: True, если benchmark-индексов ещё нет
BEST_APPROACH = None           # TODO: "text_then_geo" / "geo_then_tuple"
BEST_N = None                  # TODO: N с val
BEST_FEATURE = "none"          # TODO: none / rating / reviews / phone / messages / all
BEST_WEIGHT = 0.0              # TODO: вес с val
# Состав моделей и SCORE_ORDER берутся из проведённого сравнения.
ANSWER_PATH = Path("/kaggle/working/answer.csv")

In [ ]:
if RUN_BENCHMARK:
    if any(x is None for x in (BENCHMARK_ITEMS_PATH, BENCHMARK_QUERIES_PATH, BEST_APPROACH, BEST_N)):
        raise ValueError("Заполните пути benchmark и BEST_APPROACH/BEST_N.")
    answer = experiment.benchmark(
        items_path=BENCHMARK_ITEMS_PATH,
        queries_path=BENCHMARK_QUERIES_PATH,
        index_dir=BENCHMARK_INDEX_DIR,
        build_missing_indexes=BUILD_BENCHMARK_INDEXES,
        approach=BEST_APPROACH, n=BEST_N,
        feature=BEST_FEATURE, weight=BEST_WEIGHT,
        output_path=ANSWER_PATH,
    )
    display(answer.head())